# Lesson 3 : Language Modelling
__Teaching Machines to predict next word__

*its doing one single things at its core:

__| Given the word so for predict next word.__

The entire field of large language models is an elaborate answer to this one question.


# What is Language model ?

A Language model assigns probablities to sequence of words.

P("cat sat on the mat") = 0.0023 <- reasonable english

P("mat sat on the cat") = 0.0008 <- grammatical but weird

P("sat the on mat the cat") = 0.000001 <- garbage

__more usefully, it answers. "Given everything so far what comes next ?"__

P(next word | "The cat sat on the ___ ")

P("mat") = 0.34

P("floor") = 0.21

P("roof") = 0.08

P("dog")= 0.01

.....

## Why is this enough to build ChatGPT ?

Because if you can predict the next word well enough, you implicitly must understand :

* Grammar (to predict grammatical continuations)

* Facts (to predict factually correct continuations)

* Reasoning ( to predict logically consistent continuations)

* style (to preddict contextually appropriate continuations)

__the "predict next word" objective forces the model to learn evrything about the language__

## Chapter 1 : N-Gram Language models

the simplest possible language model

Count how often word sequences appear in training data. use those counts as probability.

__Unigram model__ - each word independent:

P("cat") = Count("cat")/total words

__Bigram model__ - Probability depends on previous 1 word:

P("sat" | "cat") = Count( "cat sat")/Count("cat")

__trigram Model__ - Probability depends on previous 2 word:

P("sat" | "the cat") = Count("sat")/Count("the cat")



In [4]:
# N-Gram Language Model from Scratch
from collections import defaultdict, Counter
import random
import math
class NgramLanguageModel:
    def __init__(self,n):
        self.n=n
        self.count=defaultdict(Counter)
        self.vocab=set()

    def train(self,text):
        words=text.lower().split()
        self.vocab.update(words)
        for i in range (len(words)-self.n+1):
            context=tuple(words[i:i+self.n-1])
            target=words[i+self.n-1]
            self.count[context][target]+=1

    def probability(self,word,context):
        context=tuple(context[-self.n-1:])
        total=sum(self.count[context].values())
        if total==0:
            return 0.0
        return (self.count[context][word]+1)/(total+len(self.vocab))
    
    def predict_next(self,context,topn=5):
        context=tuple(context[-self.n+1:])
        scores={}
        for word in self.vocab:
            scores[word]=self.probability(word,context)
        return sorted(scores.items(),key=lambda x: -x[1])[:topn]
    def generate(self,seed,max_words=20):
        words=seed.lower().split()
        for _ in range(max_words):
            predictions = self.predict_next(words)
            # Sample from top predictions (not just argmax — adds variety)
            candidates, probs = zip(*predictions)
            total_probs=sum(probs)
            if total_probs==0:
                # Fallback: If everything is 0, give every word an equal chance
                probs = [1 / len(candidates)] * len(candidates)
            else:
                probs = [p / sum(probs) for p in probs]
            next_word = random.choices(candidates, weights=probs, k=1)[0]
            words.append(next_word)
        return " ".join(words)

    def perplexity(self, text):
        """
        Perplexity = how 'surprised' is the model by this text?
        Lower = better. A perplexity of K means the model is as confused
        as if it had to choose uniformly among K words at each step.
        """
        words   = text.lower().split()
        log_prob = 0
        count    = 0
        for i in range(self.n - 1, len(words)):
            context = words[i - (self.n-1) : i]
            word    = words[i]
            p       = self.probability(word, context)
            log_prob += math.log(p + 1e-10)
            count    += 1
        return math.exp(-log_prob / count)
    




text = """the king rules the kingdom with wisdom the queen advises the king
the prince will one day rule the kingdom the princess studies ancient wisdom
the knight protects the kingdom from enemies the wizard advises with magic
the king and queen rule together with justice for all people in the kingdom
the ancient kingdom has many rules that all people must follow
the wise king listens to his queen and his wizard before making decisions
"""
# Train Bigram and Trigram models
bigram=NgramLanguageModel(n=2)
trigram=NgramLanguageModel(n=3)
bigram.train(text)
trigram.train(text)

# --- Predictions ---
print("=== BIGRAM MODEL ===")
print("Context: ['the']")
for word, prob in bigram.predict_next(["the"]):
    print(f"  P('{word}' | 'the') = {prob:.4f}")

print("\n=== TRIGRAM MODEL ===")
print("Context: ['the', 'king']")
for word, prob in trigram.predict_next(["the", "king"]):
    print(f"  P('{word}' | 'the king') = {prob:.4f}")

# --- Text generation ---
print("\n=== TEXT GENERATION ===")
print("Bigram  seed='the king':")
print(" ", bigram.generate("the king", max_words=15))

print("\nTrigram seed='the king':")
print(" ", trigram.generate("the king", max_words=15))

# --- Perplexity comparison ---
test_sentence = "the king rules the kingdom"
weird_sentence = "wizard the kingdom rules ancient"
print(f"\n=== PERPLEXITY ===")
print(f"Normal sentence:  {trigram.perplexity(test_sentence):.2f}")
print(f"Weird sentence:   {trigram.perplexity(weird_sentence):.2f}")
print("(Lower perplexity = model finds text more natural)")


=== BIGRAM MODEL ===
Context: ['the']
  P('kingdom' | 'the') = 0.0909
  P('king' | 'the') = 0.0727
  P('princess' | 'the') = 0.0364
  P('wizard' | 'the') = 0.0364
  P('knight' | 'the') = 0.0364

=== TRIGRAM MODEL ===
Context: ['the', 'king']
  P('the' | 'the king') = 0.0455
  P('rules' | 'the king') = 0.0455
  P('and' | 'the king') = 0.0455
  P('many' | 'the king') = 0.0227
  P('king' | 'the king') = 0.0227

=== TEXT GENERATION ===
Bigram  seed='the king':
  the king and queen rule together king many rules that princess princess princess princess wizard princess studies

Trigram seed='the king':
  the king king princess king king many many many many princess many many kingdom wizard wizard kingdom

=== PERPLEXITY ===
Normal sentence:  21.33
Weird sentence:   16509636.22
(Lower perplexity = model finds text more natural)


# Fatal problems with N-Gram Language Model

Run the generation few times. you'll notice.

Problem 1 : No long range dependiencies :

" The king who defeated the dragon that terrorized the village ______ "

A trigram only sees the last 2 words. It has no idea "King" was mentioned 8 words ago when it needs to predict "ruled" vs "barked".

Problem 2 : Sparsity explodes with n :

with a 50000 word vocabulary :

* Bigram table : (50,000)2 = 2.5 billion possible pairs - most never seen 
* Trigram table : (50,000)3 = 125 trillion possible triples

you need exponential more data as n grows, but you also need larger n to capture longer context. this is curse of dimensionality.

Problem 3 : No generalization.

If training data has "The king rules" but not " The emperor rules" the model has zero knowledge about emperors, even though "king~emperor" in meaning. N-grams dont use embeddings - they treat every word as completly independent symbol .

__These three problems are exactly what neural language models solve.__

# Chapter 2 : Neural Languange Models

## The Bridge : Bengio's Neural LM(2003)

Youshua Bengios 2003 paper " A Neural Probabalistic Language model" Introduced the key ideas:

1. Represent words as dense vectors embeddings(solves problem 3)
2. Use a neural network to combine them - solves problem 1&2
3. Share parameters across all positions - solves sparsity






In [ ]:
# Neural Language Model using PyTorch

import torch
import torch.nn as nn
import torch.nn.functional as F

corpus = """
the king rules the kingdom with wisdom the queen advises the king
the prince will one day rule the kingdom the princess studies ancient wisdom
the knight protects the kingdom from enemies the wizard advises with magic
the king and queen rule together with justice for all people in the kingdom
"""

words=corpus.lower().split()
vocab=set(words)
V=len(vocab)
w2i={word:i for i, word in enumerate(vocab)}
i2w={i:word for word,i in w2i.items()}

# Create training pairs for a Bigram model( input word id, target: word id)
pairs = [(w2i[words[i]],w2i[words[i+1]]) for i in range(len(words)-1)]

X=torch.tensor([p[0] for p in pairs],dtype=torch.long)
Y=torch.tensor([p[1] for p in pairs],dtype=torch.long)

print(f"Vocubalary size : {V}")
print(f"Number of training pairs : {len(pairs)}")
print(f" sample training pair :{ i2w[pairs[0][0]]} - > {i2w[pairs[0][1]]}")

# the Model
class neuralbigrammodel(nn.Module):
    def __init__(self,vocab_size,embed_dim,hidden_dim):
        super().__init__()
        # Layer 1 : word -> embedding
        self.embed=nn.Embedding(vocab_size,embed_dim)
        #Layer 2 : embedding -> hidden
        self.hidden=nn.Linear(embed_dim,hidden_dim)
        #Layer 3 : hidden -> output
        self.output=nn.Linear(hidden_dim,vocab_size)

    def forward(self,x):
        emb=self.embed(x)
        hidden=torch.tanh(self.hidden(emb))
        logits=self.output(hidden)
        return logits
    
model=neuralbigrammodel(V,embed_dim=16,hidden_dim=32)
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

# Training Loop
print("\n=== TRAINING NEURAL BIGRAM MODEL ===")
for epoch in range(500):
    logits=model(X)
    loss=F.cross_entropy(logits,Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1)%100==0:
        print(f"Epoch {epoch+1:4d}  Loss: {loss.item():.4f}  "
              f"Perplexity: {torch.exp(loss).item():.2f}")
        
# generate text
def generate(model,seed_word,w2i,i2w,max_words=20,temperature=0.8):
    model.eval()
    word_out=[seed_word]
    current=torch.tensor([w2i[seed_word]])

    with torch.no_grad():
        for _ in range(max_words):
            logits=model(current)
            # temperature scaling : higher temp = more random, lower temp = more greedy
            probs=F.softmax(logits/temperature,dim=-1)
            next_id=torch.multinomial(probs,num_samples=1).item()
            word_out.append(i2w[next_id])
            current=torch.tensor([next_id])
    return " ".join(word_out)

print("\n=== GENERATED TEXT FROM NEURAL BIGRAM MODEL ===")
for seed in ["the","king","queen"]:
    print(f"Seed: '{seed}' -> {generate(model,seed,w2i,i2w)}")


# inspecting what model has learned
print("\n=== what comes after 'king'? ===")
model.eval()
with torch.no_grad():
    logits=model(torch.tensor([w2i["king"]]))
    probs=F.softmax(logits,dim=-1).squeeze()
    topk=torch.topk(probs,k=5)
    for prob,idx in zip(topk.values,topk.indices):
        print(f"  P('{i2w[idx.item()]}' | 'king') = {prob.item():.4f}")





    









Vocubalary size : 29
Number of training pairs : 50
 sample training pair :the - > king

=== TRAINING NEURAL BIGRAM MODEL ===
Epoch  100  Loss: 0.7057  Perplexity: 2.03
Epoch  200  Loss: 0.7027  Perplexity: 2.02
Epoch  300  Loss: 0.7017  Perplexity: 2.02
Epoch  400  Loss: 0.7013  Perplexity: 2.02
Epoch  500  Loss: 0.7010  Perplexity: 2.02

=== GENERATED TEXT FROM NEURAL BIGRAM MODEL ===
Seed: 'the' -> the kingdom from enemies the kingdom with wisdom the king rules the knight protects the kingdom from enemies the king rules


KeyError: 'King'